# Phase 2: Reproducing Autoformer (Wu et al., NeurIPS 2021)

This notebook trains Autoformer with the paper's settings (Table 1, multivariate, input length 96, horizons 96 / 192 / 336 / 720)
and compares the results with the paper and with the naive baselines from the EDA.

**Run it in Colab with a GPU:** Runtime -> Change runtime type -> T4 GPU. Run the cells top to bottom.

**Where each part comes from (needed for the report's academic-integrity section):**

| Part | Source |
|---|---|
| Model, training loop, data loader, metrics | Official code, github.com/thuml/Autoformer (commit printed in cell 2) |
| Compatibility fixes for NumPy 2 / pandas 2+ (3 files) | Ours, cell 3 |
| Seed taken from an environment variable so repeated runs differ (run.py) | Ours, cell 3 |
| Experiment runner, resumable result log, comparison table, baselines, plots | Ours, cells 5 to 9 |
| Paper numbers | Wu et al. 2021, Table 1 |

In [ ]:
# 1. Check the GPU
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# 2. Get the official code and record which version was used
import os, subprocess
os.chdir("/content")
if not os.path.exists("/content/Autoformer"):
    !git clone -q https://github.com/thuml/Autoformer.git
os.chdir("/content/Autoformer")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Official Autoformer commit:", COMMIT, "<- quote this in the report")
!pip -q install reformer_pytorch      # only needed because models/__init__ imports Reformer

In [ ]:
# 3. Our patches. Each one is applied once and checked; the model and the evaluation are not changed.
import re

def patch(path, old, new, regex=False):
    s = open(path).read()
    if (re.search(new, s) if regex else new in s):
        print("already patched:", path); return
    s2 = re.sub(old, new, s) if regex else s.replace(old, new)
    assert s2 != s, f"pattern not found in {path}"
    open(path, "w").write(s2); print("patched:", path)

# NumPy 2.0 removed np.Inf (crash in EarlyStopping)
patch("utils/tools.py", "np.Inf", "np.inf")
# pandas >= 2.2 no longer accepts 't' for minutes (crash on ETTm1/ETTm2)
patch("utils/timefeatures.py", "    offset = to_offset(freq_str)",
      "    offset = to_offset({'t': 'min', 'T': 'min'}.get(freq_str, freq_str))")
# pandas 2.0 removed the positional axis argument (only hit with --embed fixed/learned)
patch("data_provider/data_loader.py", "df_stamp.drop(['date'], 1)", "df_stamp.drop(columns=['date'])")
s = open("data_provider/data_loader.py").read()
s = s.replace("df_stamp.date.apply(lambda row: row.weekday(), 1)", "df_stamp.date.dt.weekday")
s = re.sub(r"df_stamp\.date\.apply\(lambda row: row\.(month|day|hour|minute), 1\)", r"df_stamp.date.dt.\1", s)
open("data_provider/data_loader.py", "w").write(s)
# The official run.py always uses seed 2021, so "repeated" runs would start from the same weights.
patch("run.py", "    fix_seed = 2021", "    fix_seed = int(os.environ.get('SEED', 2021))")

## Data
Put the four CSVs in one Google Drive folder (default below: `MyDrive/Autoformer/data`). Results and checkpoints are saved to
`MyDrive/Autoformer/runs`, so a disconnected Colab session can carry on where it stopped.
If you do not use Drive, set `USE_DRIVE = False` and upload the files when asked.

In [ ]:
# 4. Data and output folders
import shutil, glob
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/Autoformer"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR, OUT_DIR = f"{DRIVE_DIR}/data", f"{DRIVE_DIR}/runs"
else:
    DATA_DIR, OUT_DIR = "/content/upload", "/content/runs"
    os.makedirs(DATA_DIR, exist_ok=True)
    from google.colab import files
    for name, data in files.upload().items():
        open(f"{DATA_DIR}/{name}", "wb").write(data)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("dataset", exist_ok=True)
for f in glob.glob(f"{DATA_DIR}/*.csv"):
    shutil.copy(f, "dataset/")
print("dataset/ now holds:", sorted(os.listdir("dataset")))

## Experiment settings
`DATASETS` follows the official scripts (`scripts/ETT_script`, `scripts/Exchange_script`, `scripts/ECL_script`).
Everything not listed uses the defaults in `run.py`, which match the paper: d_model 512, 8 heads, d_ff 2048, moving average 25,
2 encoder and 1 decoder layers, Adam with learning rate 1e-4, batch size 32, MSE loss, up to 10 epochs with early stopping (patience 3).

* `QUICK = True` runs a tiny model for 1 epoch, to check that everything works in a few minutes. Set it to `False` for the real runs.
* `RUNS = 3` matches the paper ("all experiments are repeated three times"). Start with 1 if GPU time is short.
* The official Exchange script trains horizons 192 and 336 for 1 epoch only; `FOLLOW_OFFICIAL_EPOCHS = True` copies that.
* The ECL file is your 10-series subset, so its numbers cannot be compared with the paper's 321-series results.

In [ ]:
# 5. Settings
QUICK = False
RUNS = 1
HORIZONS = [96, 192, 336, 720]
RUN_ORDER = ["ETTm2", "Exchange"]          # add "ECL_subset" when time allows
FOLLOW_OFFICIAL_EPOCHS = True

DATASETS = {
    "ETTm2":      dict(data="ETTm2",  data_path="ETTm2.csv",                freq="t", factor=1, n_vars=7),
    "Exchange":   dict(data="custom", data_path="exchange_rate.csv",        freq="h", factor=3, n_vars=8),
    "ECL_subset": dict(data="custom", data_path="electricity_subset10.csv", freq="h", factor=3, n_vars=10),
}
# Paper Table 1 (multivariate, input 96): {horizon: (MSE, MAE)}
PAPER = {
    "ETTm2":    {96: (0.255, 0.339), 192: (0.281, 0.340), 336: (0.339, 0.372), 720: (0.422, 0.419)},
    "Exchange": {96: (0.197, 0.323), 192: (0.300, 0.369), 336: (0.509, 0.524), 720: (1.447, 0.941)},
}

In [ ]:
# 6. Runner: one call = one training + test run. Results go to OUT_DIR/results.csv; finished runs are skipped.
import sys, time, numpy as np, pandas as pd

RESULTS_CSV = f"{OUT_DIR}/results{'_quick' if QUICK else ''}.csv"

def load_results():
    return pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()

def run_one(name, H, run):
    cfg = DATASETS[name]
    des = f"{'quick' if QUICK else 'Exp'}_run{run}"
    model_id = f"{name}_96_{H}"
    args = ["--is_training", "1", "--model", "Autoformer", "--model_id", model_id,
            "--root_path", "./dataset/", "--data", cfg["data"], "--data_path", cfg["data_path"],
            "--features", "M", "--seq_len", "96", "--label_len", "48", "--pred_len", str(H),
            "--e_layers", "2", "--d_layers", "1", "--factor", str(cfg["factor"]), "--freq", cfg["freq"],
            "--enc_in", str(cfg["n_vars"]), "--dec_in", str(cfg["n_vars"]), "--c_out", str(cfg["n_vars"]),
            "--des", des, "--itr", "1", "--num_workers", "2"]
    if QUICK:
        args += ["--d_model", "16", "--d_ff", "32", "--n_heads", "2", "--train_epochs", "1"]
    elif FOLLOW_OFFICIAL_EPOCHS and name == "Exchange" and H in (192, 336):
        args += ["--train_epochs", "1"]
    env = dict(os.environ, SEED=str(2021 + run))
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "-u", "run.py"] + args, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:                                   # show progress lines only
        if any(k in line for k in ("Epoch:", "Vali Loss", "Early stopping", "mse:", "Error", "Traceback")):
            print("   ", line.rstrip())
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"run.py failed for {model_id} run {run}; re-run this cell to see the full log")
    folder = glob.glob(f"results/{model_id}_Autoformer_*_{des}_0")[0]
    mae, mse, rmse, mape, mspe = np.load(f"{folder}/metrics.npy")
    setting = os.path.basename(folder)
    ckpt = f"checkpoints/{setting}/checkpoint.pth"
    os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
    shutil.copy(ckpt, f"{OUT_DIR}/checkpoints/{setting}.pth")   # needed later for deployment
    for f in ("pred.npy", "true.npy"):                          # large; keep only for horizon 96
        if H != 96: os.remove(f"{folder}/{f}")
    return dict(dataset=name, horizon=H, run=run, mse=float(mse), mae=float(mae),
                minutes=round((time.time() - t0) / 60, 1), setting=setting)

In [ ]:
# 7. Train everything in RUN_ORDER (safe to re-run after a disconnect: finished runs are skipped)
for name in RUN_ORDER:
    for H in HORIZONS:
        for run in range(RUNS):
            done = load_results()
            if len(done) and ((done.dataset == name) & (done.horizon == H) & (done.run == run)).any():
                continue
            print(f"== {name}  horizon {H}  run {run}")
            row = run_one(name, H, run)
            pd.concat([done, pd.DataFrame([row])]).to_csv(RESULTS_CSV, index=False)
            print(f"   MSE {row['mse']:.3f}  MAE {row['mae']:.3f}  ({row['minutes']} min)")
display(load_results())

In [ ]:
# 8. Naive baselines on exactly the same test windows (same split and train-only scaling as the official loader)
from numpy.lib.stride_tricks import sliding_window_view

def split_sizes(name, n):
    if name == "ETTm2":
        return 12 * 30 * 24 * 4, 4 * 30 * 24 * 4, 4 * 30 * 24 * 4
    n_train, n_test = int(n * 0.7), int(n * 0.2)
    return n_train, n - n_train - n_test, n_test

def baselines(name, H, seq=96):
    df = pd.read_csv(f"dataset/{DATASETS[name]['data_path']}").drop(columns="date")
    n_train, n_val, n_test = split_sizes(name, len(df))
    x = df.values[: n_train + n_val + n_test]
    mu, sd = x[:n_train].mean(0), x[:n_train].std(0)            # StandardScaler (population std), train part only
    x = (x - mu) / sd
    w = sliding_window_view(x[n_train + n_val - seq:], seq + H, axis=0)
    inp, y = w[:, :, :seq], w[:, :, seq:]
    out = {"last value": np.repeat(inp[:, :, -1:], H, axis=2)}
    if name == "ETTm2":
        out["repeat last day"] = np.tile(inp[:, :, -96:], (1, 1, int(np.ceil(H / 96))))[:, :, :H]
    if name == "ECL_subset":
        out["repeat last day"] = np.tile(inp[:, :, -24:], (1, 1, int(np.ceil(H / 24))))[:, :, :H]
    return {k: (float(np.mean((p - y) ** 2)), float(np.mean(np.abs(p - y)))) for k, p in out.items()}

BASE = {(n, H): baselines(n, H) for n in RUN_ORDER for H in HORIZONS}

In [ ]:
# 9. Comparison table: our Autoformer (mean +/- std over runs) vs the paper vs the best naive baseline
res = load_results()
rows = []
for (name, H), g in res.groupby(["dataset", "horizon"]):
    best_name, (b_mse, b_mae) = min(BASE[(name, H)].items(), key=lambda kv: kv[1][0])
    p = PAPER.get(name, {}).get(H, (np.nan, np.nan))
    rows.append({"dataset": name, "horizon": H, "runs": len(g),
                 "ours MSE": f"{g.mse.mean():.3f} ± {g.mse.std(ddof=0):.3f}", "paper MSE": p[0],
                 "ours MAE": f"{g.mae.mean():.3f} ± {g.mae.std(ddof=0):.3f}", "paper MAE": p[1],
                 "MSE gap vs paper": f"{100 * (g.mse.mean() - p[0]) / p[0]:+.1f}%" if p[0] == p[0] else "",
                 "best baseline": best_name, "baseline MSE": round(b_mse, 3), "baseline MAE": round(b_mae, 3),
                 "min per run": g.minutes.mean()})
table = pd.DataFrame(rows)
table.to_csv(f"{OUT_DIR}/comparison{'_quick' if QUICK else ''}.csv", index=False)
display(table)

In [ ]:
# 10. Look at one forecast (horizon 96, run 0): last variable (OT for ETTm2), first test window and one from the middle
import matplotlib.pyplot as plt
res = load_results()
for name in RUN_ORDER:
    r = res[(res.dataset == name) & (res.horizon == 96) & (res.run == 0)]
    if r.empty: continue
    folder = f"results/{r.setting.iloc[0]}"
    if not os.path.exists(f"{folder}/pred.npy"):
        print(name, ": predictions are not in this Colab session (re-run that experiment to plot)"); continue
    pred, true = np.load(f"{folder}/pred.npy"), np.load(f"{folder}/true.npy")
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
    for a, i in zip(ax, [0, len(pred) // 2]):
        a.plot(true[i, :, -1], label="actual"); a.plot(pred[i, :, -1], label="Autoformer")
        a.set_title(f"{name}: test window {i}, last variable (scaled)"); a.legend()
    plt.tight_layout(); plt.show()

## What to record for the report
* The commit printed in cell 2, the patches in cell 3, and the GPU type from cell 1.
* The comparison table (saved as `runs/comparison.csv` in Drive) and the minutes per run (useful for the computational-analysis experiment).
* Any gap to the paper. Small gaps are normal: the paper averages 3 runs on a TITAN RTX, and GPU kernels are not fully deterministic.
* Checkpoints are saved to `runs/checkpoints/` in Drive; the horizon-96 ETTm2 model is the natural one to deploy.